# Multi-Label Classification with Independent Sigmoids
**Task ID:** `logreg_lvl6_multilabel`  
**Series:** Logistic Regression, Level 6  
**Protocol:** `pytorch_task_v1`

## Math
- **Why sigmoid instead of softmax?**
  - Softmax: $p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$ — probabilities sum to 1 (mutually exclusive classes)
  - Sigmoid: $p_k = \frac{1}{1 + e^{-z_k}}$ — each label is an **independent** binary decision
- Model: `nn.Linear(4, 3)` produces 3 independent logits
- Loss: `BCEWithLogitsLoss` = numerically stable sigmoid + BCE combined
  $$L = -\frac{1}{NK} \sum_{i,k} \left[ y_{ik} \log \sigma(z_{ik}) + (1-y_{ik}) \log(1-\sigma(z_{ik})) \right]$$
- At inference: `predicted_labels = (sigmoid(logits) > 0.5).float()`

In [ ]:
import sys
import os
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print(f"PyTorch version: {torch.__version__}")

## Required Functions (`pytorch_task_v1` protocol)

In [ ]:
def get_task_metadata():
    return {
        "task_id": "logreg_lvl6_multilabel",
        "algorithm": "Logistic Regression (Multi-Label with Independent Sigmoids)",
        "series": "Logistic Regression",
        "level": 6,
        "interface_protocol": "pytorch_task_v1",
    }

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

print(f"Device: {get_device()}")
print(f"Task: {get_task_metadata()['task_id']}")

In [ ]:
def make_dataloaders(cfg):
    n = cfg["n_samples"]
    batch_size = cfg["batch_size"]
    val_ratio = cfg.get("val_ratio", 0.2)

    # 4 features from N(0,1)
    X = torch.randn(n, 4)

    # 3 binary labels — NOT mutually exclusive (overlapping thresholds)
    label_0 = ((X[:, 0] + X[:, 1]) > 0).float()
    label_1 = ((X[:, 1] + X[:, 2]) > 0.5).float()
    label_2 = ((X[:, 2] + X[:, 3]) < -0.2).float()
    y = torch.stack([label_0, label_1, label_2], dim=1)

    multi = (y.sum(dim=1) > 1).sum().item()
    print(f"Samples with 2+ active labels: {multi}/{n} ({100*multi/n:.1f}%)")

    n_val = int(n * val_ratio)
    idx = torch.randperm(n)
    train_idx, val_idx = idx[n_val:], idx[:n_val]

    train_ds = TensorDataset(X[train_idx], y[train_idx])
    val_ds = TensorDataset(X[val_idx], y[val_idx])

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size),
    )

In [ ]:
def build_model(cfg):
    return nn.Linear(cfg["n_features"], cfg["n_labels"])

def train(model, train_loader, val_loader, cfg, device):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])
    epochs = cfg["epochs"]
    loss_history = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(X_b)
        loss_history.append(epoch_loss / len(train_loader.dataset))

        if (epoch + 1) % 50 == 0:
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for X_b, y_b in val_loader:
                    X_b, y_b = X_b.to(device), y_b.to(device)
                    val_loss += criterion(model(X_b), y_b).item() * len(X_b)
            val_loss /= len(val_loader.dataset)
            print(f"Epoch {epoch+1}/{epochs} | Train: {loss_history[-1]:.5f} | Val: {val_loss:.5f}")

    return loss_history

In [ ]:
def _per_label_metrics(preds_bin, targets, n_labels):
    metrics = {}
    for k in range(n_labels):
        tp = ((preds_bin[:, k] == 1) & (targets[:, k] == 1)).sum().float().item()
        fp = ((preds_bin[:, k] == 1) & (targets[:, k] == 0)).sum().float().item()
        fn = ((preds_bin[:, k] == 0) & (targets[:, k] == 1)).sum().float().item()
        prec = tp / (tp + fp + 1e-8)
        rec = tp / (tp + fn + 1e-8)
        f1 = 2 * prec * rec / (prec + rec + 1e-8)
        metrics[f"label_{k}"] = {"precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4)}
    return metrics

def evaluate(model, loader, device, n_labels=3):
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            all_logits.append(model(X_b.to(device)).cpu())
            all_targets.append(y_b)
    logits = torch.cat(all_logits)
    targets = torch.cat(all_targets)
    preds_bin = (torch.sigmoid(logits) > 0.5).float()

    hamming_acc = (preds_bin == targets).float().mean().item()
    exact_match = (preds_bin == targets).all(dim=1).float().mean().item()
    per_label = _per_label_metrics(preds_bin, targets, n_labels)

    return {
        "hamming_accuracy": round(hamming_acc, 4),
        "exact_match_ratio": round(exact_match, 4),
        "per_label_metrics": per_label,
    }

def predict(model, X, device):
    model.eval()
    with torch.no_grad():
        logits = model(X.to(device)).cpu()
        return (torch.sigmoid(logits) > 0.5).float()

def save_artifacts(outputs, cfg):
    out_dir = cfg.get("output_dir", "./output")
    os.makedirs(out_dir, exist_ok=True)
    saveable = {k: v for k, v in outputs.items() if isinstance(v, (int, float, str, dict, list, bool))}
    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(saveable, f, indent=2, default=str)
    print(f"Metrics saved to {out_dir}")

## Configuration

In [ ]:
cfg = {
    "n_samples": 1000,
    "n_features": 4,
    "n_labels": 3,
    "batch_size": 32,
    "val_ratio": 0.2,
    "epochs": 200,
    "lr": 0.01,
    "hamming_acc_threshold": 0.75,
    "f1_threshold": 0.60,
    "output_dir": "./output/logreg_lvl6_multilabel",
}

## Train

In [ ]:
set_seed(42)
device = get_device()

train_loader, val_loader = make_dataloaders(cfg)
model = build_model(cfg).to(device)

print("\n--- Training ---")
loss_history = train(model, train_loader, val_loader, cfg, device)

## Evaluate

In [ ]:
train_metrics = evaluate(model, train_loader, device, cfg["n_labels"])
val_metrics = evaluate(model, val_loader, device, cfg["n_labels"])

print(f"Train | Hamming Acc: {train_metrics['hamming_accuracy']:.4f} | Exact Match: {train_metrics['exact_match_ratio']:.4f}")
print(f"Val   | Hamming Acc: {val_metrics['hamming_accuracy']:.4f} | Exact Match: {val_metrics['exact_match_ratio']:.4f}")

print("\nPer-Label Metrics (Val):")
for label, m in val_metrics["per_label_metrics"].items():
    print(f"  {label}: Precision={m['precision']:.4f}  Recall={m['recall']:.4f}  F1={m['f1']:.4f}")

## Save Artifacts & Quality Assertions

In [ ]:
outputs = {
    "loss_history": loss_history,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics,
}
save_artifacts(outputs, cfg)

print("\n" + "=" * 60)
print("Quality Assertions...")
print("=" * 60)

assert val_metrics["hamming_accuracy"] >= cfg["hamming_acc_threshold"], \
    f"Hamming accuracy {val_metrics['hamming_accuracy']:.4f} < {cfg['hamming_acc_threshold']}"
print(f"PASS: Hamming accuracy {val_metrics['hamming_accuracy']:.4f} >= {cfg['hamming_acc_threshold']}")

for label, m in val_metrics["per_label_metrics"].items():
    assert m["f1"] >= cfg["f1_threshold"], \
        f"{label} F1={m['f1']:.4f} < {cfg['f1_threshold']}"
    print(f"PASS: {label} F1={m['f1']:.4f} >= {cfg['f1_threshold']}")

print("\n=== ALL CHECKS PASSED ===")
exit_code = 0
print(f"sys.exit({exit_code})")